In [26]:
from icoder.scanner import scan_project, find_untested_functions

In [27]:
scan_result = scan_project(r"C:\Users\sasasa\Documents\Projects\icoder\sandbox")

In [28]:
scan_result

ScanResult(missing_docstrings=[FunctionInfo(name='multiply', file_path='C:\\Users\\dndlssardar\\Documents\\Projects\\icoder\\sandbox\\file1.py', lineno=17, source='    def multiply(a: float, b: float) -> float:\n        return a * b', has_docstring=False, is_async=False, decorators=['staticmethod'], class_name='Calculator'), FunctionInfo(name='multiply', file_path='C:\\Users\\dndlssardar\\Documents\\Projects\\icoder\\sandbox\\file1.py', lineno=17, source='    def multiply(a: float, b: float) -> float:\n        return a * b', has_docstring=False, is_async=False, decorators=['staticmethod'], class_name=None)], missing_tests=[], skipped_files=[])

In [29]:
docstring =  """
Multiply two numbers

Args:
    a: the first number
    b: the second number

Returns:
    the product of two numbers
"""
docstring

'\nMultiply two numbers\n\nArgs:\n    a: the first number\n    b: the second number\n\nReturns:\n    the product of two numbers\n'

In [40]:
import libcst as cst

def leave_FunctionDef2(self, original_node, updated_node):
    name = updated_node.name.value
    if name not in self.insertions:
        return updated_node
    # Check it truly has no docstring already
    body = updated_node.body
    if isinstance(body, cst.IndentedBlock):
        first = body.body[0]
        if isinstance(first, cst.SimpleStatementLine):
            if isinstance(first.body[0], cst.Expr):
                if isinstance(first.body[0].value, (cst.SimpleString, cst.ConcatenatedString, cst.FormattedString)):
                    return updated_node  # already has docstring, skip
    
    docstring_text = self.insertions[name]
    # TODO: calculate correct indentation for multi-line docstrings
    indent = "   "
    docstring_text = '"""%s\n"""' % docstring_text
    docstring_text = docstring_text.replace("\n", "\n" + indent)
    print(type(body.body[0]), type(updated_node), updated_node, body.body[0].leading_lines)

    docstring_node = cst.SimpleStatementLine(body=[cst.Expr(value=cst.SimpleString(docstring_text))])
    new_body = body.with_changes(body=[docstring_node, *body.body])
    return updated_node.with_changes(body=new_body)

class DocstringInserter(cst.CSTTransformer):
    def __init__(self, insertions: dict):
        # insertions = {func_name: docstring_text}
        self.insertions = insertions
    
    def leave_FunctionDef(self, original_node, updated_node):
        return leave_FunctionDef2(self, original_node, updated_node)

from writer import write_docstrings

info = scan_result.missing_docstrings[0]
file_path = info.file_path
print(file_path)

insertions = {info.name:docstring}

with open(file_path, "r") as f:
    source = f.read()
tree = cst.parse_module(source)
new_tree = tree.visit(DocstringInserter(insertions))
print("-======================")
print(new_tree.code)

C:\Users\dndlssardar\Documents\Projects\icoder\sandbox\file1.py
<class 'libcst._nodes.statement.SimpleStatementLine'> <class 'libcst._nodes.statement.FunctionDef'> FunctionDef(
    name=Name(
        value='multiply',
        lpar=[],
        rpar=[],
    ),
    params=Parameters(
        params=[
            Param(
                name=Name(
                    value='a',
                    lpar=[],
                    rpar=[],
                ),
                annotation=Annotation(
                    annotation=Name(
                        value='float',
                        lpar=[],
                        rpar=[],
                    ),
                    whitespace_before_indicator=SimpleWhitespace(
                        value='',
                    ),
                    whitespace_after_indicator=SimpleWhitespace(
                        value=' ',
                    ),
                ),
                equal=MaybeSentinel.DEFAULT,
                default=None,
   